# Static-BAR1 P2P on 4× CMP 170HX behind PLX switches — works on all 12 pairs; helps TP2, hurts TP4 decode

| Metric | Value |
|---|---|
| Direct P2P | **12/12 ordered pairs**, every byte verified, 5.79 GB/s each (bare metal and inside an LXC container) |
| Copy latency, P2P vs host-staged | **16 vs 22 µs** at 4 KiB · **25 vs 41 µs** at 64 KiB · **194 vs 343 µs** at 1 MiB |
| Large copies | 5.79 GB/s P2P vs 6.27 GB/s staged (P2P −8 %) |
| Qwen3.8-27B W4A16 TP2, 1 user | **80.0 vs 71.8 tok/s** (+11 %); with MTP k=3 **110.8 vs 97.6** (+14 %) |
| GLM-5.3-Flash TP4, 1 user structured | P2P on **293** vs off **396 tok/s** (−26 %) — leave P2P off for TP4 here |
| What it took | serialized BAR1 resize + a hand-programmed PCIe layout (BAR3 below each 64 GiB BAR1) adopted via kexec |

Earlier attempt on the same host, without P2P: [`2026-10-01-cmp170hx-3card-p2p-plx`](../results/2026-10-01-cmp170hx-3card-p2p-plx/README.md).
Evidence: [`results/2026-10-01-cmp170hx-4card-bar1-p2p-plx/`](../results/2026-10-01-cmp170hx-4card-bar1-p2p-plx/README.md)

![topology](../assets/charts/2026-10-01-cmp170hx-4card-bar1-p2p-plx-1-topology.png)

In [1]:
# --- Status cell -------------------------------------------------------
# Replays the committed receipts under results/<experiment>/ and re-renders the charts.
# The measurements need the patched driver + layout described in section 3.
import os, sys, json, statistics, importlib.util
EXPERIMENT = "2026-10-01-cmp170hx-4card-bar1-p2p-plx"
RESULTS_DIR = os.path.join("..", "results", EXPERIMENT)
print("receipts:", RESULTS_DIR)

receipts: ../results/2026-10-01-cmp170hx-4card-bar1-p2p-plx


In [2]:
from IPython.display import display, Markdown, Image


def load(*p):
    with open(os.path.join(RESULTS_DIR, "receipts", *p)) as f:
        return json.load(f)


def table(headers, rows):
    out = ["| " + " | ".join(headers) + " |", "|" + "|".join(["---"] * len(headers)) + "|"]
    out += ["| " + " | ".join(str(c) for c in r) + " |" for r in rows]
    display(Markdown("\n".join(out)))


spec = importlib.util.spec_from_file_location("charts", os.path.join(RESULTS_DIR, "tools", "charts.py"))
charts = importlib.util.module_from_spec(spec)
spec.loader.exec_module(charts)
env = json.load(open(os.path.join(RESULTS_DIR, "environment.json")))
table(["Item", "Value"], [(k, v) for k, v in env.items()])

| Item | Value |
|---|---|
| measured | 2026-10-01 |
| cards | 4x NVIDIA CMP 170HX (GA100, SM80), memory-unlocked to 65536 MiB each, BAR1 64 GiB each |
| topology | all four on CPU2; GPU0+GPU1 behind PLX PEX 8747 switch A, GPU2+GPU3 behind PEX 8747 switch B; Gen2 x16 per card, switch uplinks Gen3 x16 |
| host | dual Xeon E5-2686 v4 (Broadwell), Proxmox VE 9 (Debian 13), kernel 7.0.2, bare metal; LXC container for the GLM runs and one verify run |
| kernel_cmdline_pci | intel_iommu=on iommu=pt pci=realloc |
| bios | Above 4G decoding enabled, MMIO high size 1024G, MMIOH base 2T |
| driver | NVIDIA open kernel modules 615.71.09 + upstream cmpunlocker (amoghmunikote/cmpunlocker @ 88e39ce) + patches/rebar-serialize.patch + the four BAR1-P2P patches from admunch888/cmpunlocker (p2p-caps-override, p2p-bar1, p2p-skip-mailbox-preinit, p2p-readcap-override), built against the upstream tree |
| p2p_regkeys | NVreg_RegistryDwords=RMForceStaticBar1=1;RMPcieP2PType=1;RmForceEnableGen2=1;RMPcieLinkSpeed=0x1 |
| bar_layout | tools/cmp-bar-layout.py --unlock before the driver loads (setpci BAR1 unlock + ReBAR 64 GiB + hand-programmed BARs/bridge windows), then kexec into the same kernel so it adopts the layout; tools/boot/ automates it per boot |
| power_limit_w | 180 |
| copy_tools | tools/p2p_latency.py (--peer enables cuCtxEnablePeerAccess), p2p-verify.py from admunch888/cmpunlocker tools/ |
| nccl_runtime | ghcr.io/pixelml/club-170hx:vllm-glm53-sm80-pp-20260905 (torch 2.13.0+cu130, NCCL 2.29.7) |
| qwen_runtime | same image, vLLM, Qwen3.8-27B W4A16 AutoRound, TP2, 180 W, no prefix cache |
| glm_runtime | Morrowmake/glm53-flash-cmp170hx-recipe release 1.6.0 (image by digest), TP4, DFlash2 adaptive depth, 262144 ctx, 180 W |

## 1. TL;DR

- **P2P works on all four cards.** `cuCtxEnablePeerAccess` succeeds on every pair and a keyed
  256 MiB copy lands byte-exact on all 12 ordered pairs, same-switch and across switches, on
  bare metal and from inside an LXC container sharing the host driver.
- **It is a latency win, not a bandwidth win.** Small and medium copies are 1.4–1.8× faster;
  above 16 MiB the host-staged path is slightly faster (6.27 vs 5.79 GB/s).
- **Workload result splits.** Qwen3.8-27B TP2 gains 11–14 % single-user decode (vLLM switches to
  its custom all-reduce). GLM-5.3-Flash TP4 *loses* 25–44 %: with P2P off vLLM uses a faster
  host-staged all-reduce on this PLX + Broadwell path, and matches the published P2P-off numbers.
- **Getting there** needed two fixes beyond the published P2P patches: a serialized BAR1 resize
  (parallel resizes behind one switch corrupted the kernel) and a hand-programmed PCIe layout,
  because the BIOS leaves no room for two 64 GiB BAR1s behind one switch.

## 2. Results (measured)

### 2.1 Why two cards behind one switch had no BAR1 — and the layout that fixes it

![address map](../assets/charts/2026-10-01-cmp170hx-4card-bar1-p2p-plx-2-address-map.png)

Each card needs a 64 GiB BAR1 for static-BAR1 P2P. The BIOS puts switch A's window at the very
top of CPU2's 1 TiB 64-bit aperture, so it cannot grow. A kernel remove/rescan sizes the window
for two 64 GiB BARs but places each card's 32 MiB BAR3 right after its BAR1, leaving the second
64 GiB-aligned slot 32 MiB short. Placing BAR3 *below* each BAR1 fits both.

In [3]:
for name in ("layout-before.txt", "layout-after.txt"):
    print("=" * 20, name)
    print(open(os.path.join(RESULTS_DIR, "receipts", name)).read())

==================== layout-before.txt
# 64-bit prefetchable windows, CPU2 root bus (aperture 0x30000000000-0x3ffffffffff = 1 TiB), from lspci/proc-iomem
# (1) BIOS + kernel default, upstream driver (no P2P), 3 cards, 2026-10-01 13:05 GMT+7
root port A (switch A, 2 cards): 0x3ffe8000000-0x3fff5ffffff  224 MiB  (top of the aperture: cannot grow upward)
  card on port A1: BAR1 0x3fff0000000 64 MiB (fell back), BAR3 32 MiB
  card on port A2: BAR1 0x3ffe8000000 64 MiB (fell back), BAR3 32 MiB
root port B (switch B, 1 card): 0x31000000000-0x32001ffffff  64 GiB + 32 MiB (relocated: single child)
  card on port B1: BAR1 0x31000000000 64 GiB, BAR3 0x32000000000 32 MiB
# static-BAR1 P2P driver (fork) on the same layout: second card behind switch A
#   "BAR 1 [mem size 0x1000000000]: can't assign; no space" (and 32G..8G retries) -> no BAR1 -> probe-time init -EIO every 12 s
# (2) setpci unlock + ReBAR 64 GiB on all cards, remove + rescan of both root ports (kernel re-layout), 3 cards
root port A

### 2.2 Copy bandwidth and latency: P2P vs host-staged

![bandwidth and latency](../assets/charts/2026-10-01-cmp170hx-4card-bar1-p2p-plx-3-bandwidth-latency.png)

In [4]:
st, d2h = charts.latency("p2p-latency-stock.json")
pp, _ = charts.latency("p2p-latency-4card-peer.json")
nopeer, _ = charts.latency("p2p-latency-bar1.json")
table(["Size", "staged (stock) µs", "P2P driver, peer access off µs", "P2P µs", "D2H ref µs", "P2P speed-up"],
      [(charts.human(s), f"{st[s]:.1f}", f"{nopeer[s]:.1f}", f"{pp[s]:.1f}", f"{d2h[s]:.1f}", f"{st[s] / pp[s]:.2f}×") for s in sorted(st)])

| Size | staged (stock) µs | P2P driver, peer access off µs | P2P µs | D2H ref µs | P2P speed-up |
|---|---|---|---|---|---|
| 4 KiB | 22.0 | 21.8 | 15.8 | 9.5 | 1.39× |
| 64 KiB | 40.2 | 40.7 | 25.0 | 18.4 | 1.61× |
| 1 MiB | 342.9 | 340.0 | 194.0 | 165.3 | 1.77× |
| 4 MiB | 833.8 | 833.9 | 739.4 | 634.8 | 1.13× |
| 16 MiB | 2829.9 | 2826.8 | 2913.9 | 2521.5 | 0.97× |
| 64 MiB | 10819.5 | 10803.2 | 11604.9 | 10043.6 | 0.93× |
| 256 MiB | 42802.8 | 42702.4 | 46368.4 | 40171.3 | 0.92× |

The middle column is the P2P-capable driver with peer access **not** enabled: it matches the stock
driver, so `cuMemcpyPeer` still stages through host memory until `cuCtxEnablePeerAccess` is called.

### 2.3 All 12 pairs, and same switch vs across switches

![pairs](../assets/charts/2026-10-01-cmp170hx-4card-bar1-p2p-plx-4-pairs-switch.png)

In [5]:
print(open(os.path.join(RESULTS_DIR, "receipts", "p2p-verify-4card-host.txt")).read())

# tools/p2p-verify.py (from admunch888/cmpunlocker tools/), 4 cards, static-BAR1 P2P driver, bare-metal host
# console capture, 2026-10-01 ~14:55 GMT+7; 256 MiB per ordered pair, destination checked byte-for-byte on the host
Enabling peer access: all 12 ordered pairs enabled
pair         result     bandwidth    detail
[0]->[1]     OK         5.79 GB/s    all 268435456 bytes verified
[0]->[2]     OK         5.79 GB/s    all 268435456 bytes verified
[0]->[3]     OK         5.79 GB/s    all 268435456 bytes verified
[1]->[0]     OK         5.79 GB/s    all 268435456 bytes verified
[1]->[2]     OK         5.79 GB/s    all 268435456 bytes verified
[1]->[3]     OK         5.79 GB/s    all 268435456 bytes verified
[2]->[0]     OK         5.79 GB/s    all 268435456 bytes verified
[2]->[1]     OK         5.79 GB/s    all 268435456 bytes verified
[2]->[3]     OK         5.79 GB/s    all 268435456 bytes verified
[3]->[0]     OK         5.79 GB/s    all 268435456 bytes verified
[3]->[1]     OK     

### 2.4 NCCL all-reduce

![nccl](../assets/charts/2026-10-01-cmp170hx-4card-bar1-p2p-plx-5-nccl.png)

With P2P on, NCCL moves 2-GPU traffic over `P2P/CUMEM` instead of shared memory: same-switch
bus bandwidth rises from 3.7 to 4.8 GB/s. Three GPUs (one pair cross-switch) gain less and fall
behind SHM at large sizes.

In [6]:
rows = []
for name, lab in (("nccl-0,1.log", "2 GPU SHM"), ("nccl-p2p-0,1.log", "2 GPU P2P"), ("nccl-0,1,2.log", "3 GPU SHM"), ("nccl-p2p-0,1,2.log", "3 GPU P2P")):
    d = charts.nccl(name)
    for b in (8192, 1 << 20, 1 << 28):
        rows.append((lab, charts.human(b), f"{d[b]['median_us']:.0f}", f"{d[b]['busbw_GBps']:.2f}"))
table(["Run", "Size", "µs", "busbw GB/s"], rows)

| Run | Size | µs | busbw GB/s |
|---|---|---|---|
| 2 GPU SHM | 8 KiB | 144 | 0.06 |
| 2 GPU SHM | 1 MiB | 393 | 2.67 |
| 2 GPU SHM | 256 MiB | 72736 | 3.69 |
| 2 GPU P2P | 8 KiB | 119 | 0.07 |
| 2 GPU P2P | 1 MiB | 317 | 3.31 |
| 2 GPU P2P | 256 MiB | 55536 | 4.83 |
| 3 GPU SHM | 8 KiB | 134 | 0.08 |
| 3 GPU SHM | 1 MiB | 458 | 3.05 |
| 3 GPU SHM | 256 MiB | 84389 | 4.24 |
| 3 GPU P2P | 8 KiB | 153 | 0.07 |
| 3 GPU P2P | 1 MiB | 483 | 2.90 |
| 3 GPU P2P | 256 MiB | 92732 | 3.86 |

### 2.5 Qwen3.8-27B W4A16, TP2 (vLLM sm80, 180 W)

![qwen tp2](../assets/charts/2026-10-01-cmp170hx-4card-bar1-p2p-plx-6-qwen-tp2.png)

With P2P vLLM uses its custom all-reduce (`['CUSTOM', 'PYNCCL']`) instead of PyNCCL only.
Single-user decode gains 11 % (cross-switch) and 8 % (same switch); prefill moves ±2 %. Under
load the same-switch pair loses ground (−18 % at 16 users): that pair includes the hottest card,
so part of it may be thermal, not P2P.

In [7]:
rows = []
for lab, off, on in (("cross", "qwen-tp2-cross", "qwen-tp2-cross-p2p"), ("same", "qwen-tp2-same", "qwen-tp2-same-p2p"), ("cross + MTP3", "qwen-tp2-cross-mtp3", "qwen-tp2-cross-mtp3-p2p")):
    s0, c0 = charts.qwen(off)
    s1, c1 = charts.qwen(on)
    rows.append((lab, f"{s0['decode900'][0]:.1f} → {s1['decode900'][0]:.1f}", f"{s0['prefill_long'][1]:.0f} → {s1['prefill_long'][1]:.0f}",
                 f"{c0[16]:.0f} → {c1[16]:.0f}"))
table(["TP2 pair", "decode tok/s (no P2P → P2P)", "prefill tok/s", "aggregate @16"], rows)

| TP2 pair | decode tok/s (no P2P → P2P) | prefill tok/s | aggregate @16 |
|---|---|---|---|
| cross | 71.8 → 80.0 | 1738 → 1695 | 535 → 557 |
| same | 71.5 → 76.8 | 1695 → 1778 | 530 → 436 |
| cross + MTP3 | 97.2 → 110.8 | 1658 → 1641 | 385 → 359 |

### 2.6 GLM-5.3-Flash TP4 (Morrowmake recipe 1.6.0, DFlash2, 180 W): P2P hurts here

![glm tp4](../assets/charts/2026-10-01-cmp170hx-4card-bar1-p2p-plx-7-glm-tp4.png)

![glm step](../assets/charts/2026-10-01-cmp170hx-4card-bar1-p2p-plx-8-glm-step.png)

Same drafter acceptance (6.69 tokens per step at one user) in every run, so the difference is
pure step time: **19.4 ms with P2P off vs 26.1 ms with P2P on**. TP4 decode issues about a hundred
small collectives per step; on this host the host-staged all-reduce vLLM selects without P2P is
faster than the BAR1 path through the PLX switches and the Broadwell root complex. With P2P off
the box matches the recipe's published P2P-off numbers (396 / 306 / 193 tok/s single user,
799 / 720 / 516 at eight users, vs 394 / 377 / 181 and 798 / 683 / 545). The recipe's P2P gain
(437 tok/s) was measured on cards on EPYC root ports without PLX switches. Replicated embedding
adds ~1 %; the `1stage` all-reduce is much worse. Full GLM write-up: separate notebook.

In [8]:
rows = []
for run, lab, _ in charts.glm_runs():
    r = [lab]
    for ph in ("structured-c1", "coding-c1", "prose-c1"):
        r.append(f"{charts.glm(run, ph)['tok_s_median']:.1f}")
    for ph in ("structured-c8", "coding-c8", "prose-c8"):
        r.append(f"{charts.glm(run, ph)['aggregate_tok_s_median']:.0f}")
    r.append(f"{charts.glm(run, 'structured-c1')['decode_ms_per_draft_step_median']:.1f}")
    rows.append(r)
table(["Config", "1u struct", "1u code", "1u prose", "8u struct", "8u code", "8u prose", "ms/step 1u"], rows)

| Config | 1u struct | 1u code | 1u prose | 8u struct | 8u code | 8u prose | ms/step 1u |
|---|---|---|---|---|---|---|---|
| P2P off | 396.0 | 306.2 | 192.5 | 799 | 720 | 516 | 19.4 |
| P2P off + replicated embed | 398.4 | 307.3 | 192.9 | 812 | 722 | 519 | 19.3 |
| P2P on | 293.4 | 228.7 | 154.3 | 451 | 378 | 320 | 26.1 |
| P2P on + replicated embed | 296.0 | 230.4 | 155.3 | 453 | 381 | 321 | 25.9 |
| P2P on + 1stage all-reduce | 214.4 | 167.9 | 109.7 | 242 | 206 | 181 | 35.8 |

### 2.7 Where P2P pays

![summary](../assets/charts/2026-10-01-cmp170hx-4card-bar1-p2p-plx-9-summary.png)

In [9]:
for f in charts.ALL:
    f()  # re-render every chart from the receipts
print("charts re-rendered:", len(charts.ALL))

charts re-rendered: 9


## 3. Reproduce

**Hardware.** Two or more memory-unlocked CMP 170HX at x16, forced airflow, live temperature
monitoring. The layout step is board-specific: it hard-codes this host's bus numbers.

1. Driver: upstream cmpunlocker + [`patches/rebar-serialize.patch`](../results/2026-10-01-cmp170hx-4card-bar1-p2p-plx/patches/rebar-serialize.patch)
   + the four BAR1-P2P patches from admunch888/cmpunlocker (`driver/patches/p2p-*.patch`), declared in
   the build's patch order and `common/constants.yaml`; install with `--no-iommu --no-gen2-service`.
2. Layout, with no driver bound: edit `CARDS`/`UPSTREAM` in [`tools/cmp-bar-layout.py`](../results/2026-10-01-cmp170hx-4card-bar1-p2p-plx/tools/cmp-bar-layout.py)
   for your board, `--dry-run`, then `--unlock`, then `kexec -l /boot/vmlinuz-$(uname -r) --initrd=/boot/initrd.img-$(uname -r) --reuse-cmdline && systemctl kexec`.
3. Load the driver with `NVreg_RegistryDwords="RMForceStaticBar1=1;RMPcieP2PType=1;..."` and check
   `nvidia-smi -q -d MEMORY` shows 65536 MiB BAR1 on every card.
4. Verify with a keyed copy test (not `canAccessPeer` alone): `p2p-verify.py` from admunch888/cmpunlocker,
   then [`tools/p2p_latency.py --peer`](../results/2026-10-01-cmp170hx-4card-bar1-p2p-plx/tools/p2p_latency.py).
5. Per-boot automation: [`tools/boot/`](../results/2026-10-01-cmp170hx-4card-bar1-p2p-plx/tools/boot/) (modprobe guard in the initramfs +
   a oneshot unit that programs the layout and kexecs once, falling back to non-P2P on any mismatch).

For TP4 GLM-5.3-Flash leave `VLLM_ALLOW_PCIE_P2P_CUSTOM_ALLREDUCE=0` on hosts like this one.

## 4. Appendix

<details>
<summary>Limits and open items</summary>

- Only the CPU2 switch pair was tested; the CPU1 switch board's link was down during these runs.
- The same-switch Qwen concurrency loss overlaps with the hottest card (GPU0 reached 82–85 °C in
  some runs); not separated from P2P effects.
- GLM numbers use the recipe's default DFlash2 drafter (CC BY-NC-ND 4.0) as a benchmark only; an
  Apache-2.0 drop-in drafter failed to load on this engine (KV page-size mismatch) and is open.
- `p2p-verify` outputs are console captures; the latency receipts are JSON from `tools/p2p_latency.py`.
</details>